# Sicurre — CamemBERTav2 Training
**Dataset sources required on Kaggle:**
- `sicurre-ml-src` — versioned copy of `src/` from this repo (uploaded by CI)
- `sicurre-data` — frozen training CSV splits (synced from Cloudflare R2)

In [ ]:
# ── Cell 0: Environment setup ─────────────────────────────────────────
import importlib.util
import subprocess, sys, os
from pathlib import Path

# 0a. Detect accelerator FIRST — determines which fixes are needed.
_on_tpu = importlib.util.find_spec("torch_xla") is not None
print(f"[setup] Accelerator : {'TPU (torch_xla present)' if _on_tpu else 'GPU / CPU'}")

# 0b. TPU-only: fix the tensorflow / torch_xla conflict.
#     On GPU kernels tensorflow is not installed alongside XLA so this is skipped.
if _on_tpu:
    print("[setup] TPU: uninstalling tensorflow, installing tensorflow-cpu...")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow"],
        capture_output=True,   # suppress noise if not installed
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--root-user-action=ignore", "tensorflow-cpu"],
        check=True,
    )
    print("[setup] TPU: tensorflow-cpu installed.")

# 0c. Show full 4-level tree of /kaggle/input so we know exact mount paths.
print("\n=== Kaggle input datasets (4 levels) ===")
kaggle_input = Path("/kaggle/input")
for lvl1 in sorted(kaggle_input.iterdir()):
    print(f"  {lvl1.name}/")
    try:
        for lvl2 in sorted(lvl1.iterdir())[:12]:
            print(f"    {lvl2.name}{'/' if lvl2.is_dir() else ''}")
            if lvl2.is_dir():
                for lvl3 in sorted(lvl2.iterdir())[:8]:
                    print(f"      {lvl3.name}{'/' if lvl3.is_dir() else ''}")
                    if lvl3.is_dir():
                        for lvl4 in sorted(lvl3.iterdir())[:8]:
                            print(f"        {lvl4.name}{'/' if lvl4.is_dir() else ''}")
    except PermissionError:
        pass
print()

# 0c2. Extract any zip archives in the input datasets.
#      CI uploads src.zip with src/ prefix preserved; Kaggle does not
#      auto-extract dataset zips, so we do it here.
import zipfile as _zipfile
_zips_found = list(sorted(kaggle_input.rglob("*.zip")))
print(f"[setup] Zip files found: {len(_zips_found)}")
for _zip in _zips_found:
    print(f"[setup]   {_zip.relative_to(kaggle_input)}")
    _dest = _zip.parent
    try:
        with _zipfile.ZipFile(_zip) as _zf:
            _members = _zf.namelist()
            print(f"[setup]   -> extracting {len(_members)} entries to {_dest}")
            _zf.extractall(_dest)
    except Exception as _e:
        print(f"[setup] WARNING: could not extract {_zip.name}: {_e}")
print("[setup] Zip extraction done.")

# 0d. Dynamically find the directory that contains src/.
_src_root = None
for _candidate in kaggle_input.rglob("src/config/training_config.py"):
    _src_root = str(_candidate.parent.parent.parent)
    break

if _src_root is None:
    raise RuntimeError(
        "Could not locate src/config/training_config.py under /kaggle/input.\n"
        "Check that the 'sicurre-ml-src' dataset is attached and its zip\n"
        "contains the src/ folder at the root level."
    )
print(f"[setup] src root    : {_src_root}")
sys.path.insert(0, _src_root)

# 0e. Dynamically find the training data directory (contains CSV splits).
_data_dir = None
for _csv in kaggle_input.rglob("train.csv"):
    _data_dir = str(_csv.parent)
    break
if _data_dir is None:
    for _csv in kaggle_input.rglob("*.csv"):
        _data_dir = str(_csv.parent)
        break

if _data_dir is None:
    print("[setup] WARNING: No training CSV files found — attach sicurre-data dataset.")
    _data_dir = "/kaggle/input/sicurre-data"
else:
    print(f"[setup] data dir    : {_data_dir}")

# 0f. Upgrade pip, then install training dependencies.
print("[setup] Upgrading pip...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip",
     "--root-user-action=ignore", "-q"],
    check=True,
)
print("[setup] Installing training dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--root-user-action=ignore",
     "accelerate>=1.1",
     "evaluate>=0.4",
     "mlflow-skinny>=3.0",
     "scipy>=1.10",
     "sentencepiece>=0.2",
    ],
    check=True,
)
print("[setup] Done.")


In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────
import warnings

# Suppress environment-specific warnings based on accelerator detected in Cell 0.
# On GPU kernels _on_tpu is False, so these filters are skipped entirely.
if globals().get("_on_tpu", False):
    warnings.filterwarnings("ignore", message=".*transparent hugepages.*")
    warnings.filterwarnings("ignore", message=".*tensorflow.*can conflict.*")
    warnings.filterwarnings("ignore", category=UserWarning, module="torch_xla")
    warnings.filterwarnings("ignore", category=UserWarning, module="jax")

print("[imports] Loading src modules...")
from pathlib import Path

import mlflow
import numpy as np

from src.config.training_config import (
    RuntimeState,
    TrainingConfig,
    detect_device,
    detect_runtime,
    load_secrets,
)
from src.data.loader import load_splits
from src.data.tokenizer import prepare_tokenized_splits
from src.evaluation import (
    build_error_dataframe,
    confusion_matrix_arrays,
    evaluate_on_test,
    save_classification_report,
)
from src.model.builder import compute_class_weights, load_model
from src.registry import promote_if_threshold, push_to_hub, register_model, setup_mlflow
from src.training.baseline import prepare_baseline_training
print("[imports] OK")


In [ ]:
# ── Cell 2: Runtime + secrets ─────────────────────────────────────────
print("[runtime] Detecting environment...")
runtime_env = detect_runtime()
device, use_tpu = detect_device()
secrets = load_secrets(runtime_env)

print(f"[runtime] Runtime : {runtime_env.upper()}")
print(f"[runtime] Device  : {device}  |  TPU: {use_tpu}")
_loaded = [k for k, v in secrets.items() if v]
print(f"[runtime] Secrets loaded: {_loaded}")


In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────
from datetime import datetime

config = TrainingConfig()

# Use the path detected in Cell 0; fall back to default if Cell 0 wasn't run.
DATA_DIR   = Path(globals().get("_data_dir", "/kaggle/input/sicurre-data"))
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

state = RuntimeState(
    runtime_env=runtime_env,
    device=device,
    use_tpu=use_tpu,
    run_date=datetime.now().strftime("%Y%m%d"),
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    secrets=secrets,
    hf_token=secrets.get("HF_TOKEN"),
    databricks_host=secrets.get("DATABRICKS_HOST"),
    databricks_token=secrets.get("DATABRICKS_TOKEN"),
    databricks_email=secrets.get("DATABRICKS_EMAIL"),
    mlflow_experiment_name=secrets.get("MLFLOW_EXPERIMENT_NAME") or "sicurre-phishing",
)

HF_USERNAME = secrets.get("HF_USERNAME", "")
REPO_NAME   = secrets.get("REPO_NAME", "sicurre-phishing-detector")
HF_REPO_ID  = f"{HF_USERNAME}/{REPO_NAME}"
MODEL_NAME  = "main.sicurre.phishing-detector"

print(f"[config] DATA_DIR   : {DATA_DIR}")
print(f"[config] OUTPUT_DIR : {OUTPUT_DIR}")
print(f"[config] HF repo    : {HF_REPO_ID}")


In [ ]:
# ── Cell 4: MLflow setup ──────────────────────────────────────────────
experiment_path = setup_mlflow(state)
print(f"Experiment : {experiment_path}")
print(f"Tracking   : {mlflow.get_tracking_uri()}")

In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────
print(f"[data] Loading splits from {DATA_DIR}")
csv_files = list(DATA_DIR.glob("*.csv"))
print(f"[data] CSV files found: {[f.name for f in csv_files]}")
if not csv_files:
    raise RuntimeError(
        f"No CSV files found in {DATA_DIR}.\n"
        f"Check that the 'sicurre-data' dataset is attached.\n"
        f"Contents: {list(DATA_DIR.iterdir())}"
    )

splits = load_splits(DATA_DIR, config.label2id)
print(f"[data] Train : {len(splits.train):,}  |  Val : {len(splits.val):,}  |  Test : {len(splits.test):,}")


In [ ]:
# ── Cell 6: Tokenize ──────────────────────────────────────────────────
print("[tokenize] Tokenizing splits...")
tok_splits = prepare_tokenized_splits(
    train_df=splits.train,
    val_df=splits.val,
    test_df=splits.test,
    model_name=config.model_name,
    max_length=config.max_length,
    hf_token=state.hf_token,
)
print(f"[tokenize] train: {len(tok_splits.train)}  val: {len(tok_splits.val)}  test: {len(tok_splits.test)}")


In [ ]:
# ── Cell 7: Model + class weights ─────────────────────────────────────
model = load_model(config, hf_token=state.hf_token)

class_weights = compute_class_weights(
    splits.train["label"].values,
    num_labels=config.num_labels,
    strategy=config.class_weight_strategy,
)

print(f"Class weights: {class_weights.tolist()}")

In [ ]:
# ── Cell 8: Baseline setup ────────────────────────────────────────────
setup = prepare_baseline_training(
    train_df=splits.train,
    tokenized_splits=tok_splits,
    config=config,
    runtime=state,
)
trainer = setup.trainer

print(f"Run name    : {setup.run_name}")
print(f"Output dir  : {setup.output_dir}")
print(f"Warmup steps: {setup.training_args.warmup_steps}")

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────
with mlflow.start_run(run_name=setup.run_name) as run:
    mlflow.log_params(setup.run_config)
    trainer.train()
    mlflow.log_metrics({k: v for k, v in trainer.state.log_history[-1].items() if isinstance(v, float)})

baseline_run_id = run.info.run_id
print(f"Run ID: {baseline_run_id}")

In [ ]:
# ── Cell 10: Evaluate on test set ─────────────────────────────────────
print("[eval] Running test set evaluation...")
test_metrics = evaluate_on_test(trainer, tok_splits.test)

# trainer.evaluate() triggers the HuggingFace MLflow callback, which may
# leave the training run active. End it before we try to resume it.
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_id=baseline_run_id):
    mlflow.log_metrics(test_metrics)

print(f"[eval] F1 weighted     : {test_metrics.get('test_f1_weighted', 0):.4f}")
print(f"[eval] Phishing recall : {test_metrics.get('test_phishing_recall', 0):.4f}")


In [ ]:
# ── Cell 11: Save model ───────────────────────────────────────────────
save_path = setup.output_dir / "best"
trainer.save_model(str(save_path))
tok_splits.tokenizer.save_pretrained(str(save_path))
print(f"Saved to {save_path}")

In [ ]:
# ── Cell 12: Error analysis + confusion matrix ────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from src.config.training_config import LABEL_NAMES

test_preds = trainer.predict(tok_splits.test)
pred_labels = np.argmax(test_preds.predictions, axis=-1)
true_labels = splits.test["label"].values

error_df = build_error_dataframe(splits.test, test_preds.predictions)
errors   = error_df[~error_df["correct"]]
print(f"Errors: {len(errors)} / {len(error_df)} ({100*len(errors)/len(error_df):.1f}%)")

cm, cm_norm = confusion_matrix_arrays(true_labels, pred_labels)
report_path = save_classification_report(true_labels, pred_labels, OUTPUT_DIR)
print(f"Report saved: {report_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
axes[1].set_title("Confusion Matrix (normalised)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 13: Register model in MLflow Unity Catalog ───────────────────
model_info = register_model(
    run_id=baseline_run_id,
    save_path=save_path,
    model_name=MODEL_NAME,
    config=config,
)
print(f"[registry] Registered: {MODEL_NAME} v{model_info.registered_model_version}")


In [ ]:
# ── Cell 14: Promote ──────────────────────────────────────────────────
promoted = promote_if_threshold(
    model_name=MODEL_NAME,
    test_metrics=test_metrics,
    recall_threshold=0.97,
    f1_threshold=0.90,
)
alias = "@production" if promoted else "@staging"
print(f"Alias assigned: {alias}")

In [ ]:
# ── Cell 15: Push to HuggingFace (only if @production) ────────────────
if promoted and state.hf_token:
    hf_url = push_to_hub(
        save_path=save_path,
        hf_repo_id=HF_REPO_ID,
        hf_token=state.hf_token,
        test_metrics=test_metrics,
        mlflow_version=str(model_info.registered_model_version),
        artifact_dir=OUTPUT_DIR,
    )
    print(f"Live: {hf_url}")
else:
    print("Skipped HF push — model did not reach @production threshold.")